# 04 — Module 2: schema and drift detection

Three steps: **structural check** (columns dropped/renamed/retyped), **statistical check** (KS test for numeric columns, Chi-squared for categorical), and a **Random Forest drift classifier** trained on the drift statistics.

### Controlled-experiment note (important)
For the Random Forest, the clean (label 0) batches are sampled from the **same period as the reference** (days 1-3). This isolates injected drift as the only signal. If clean batches came from a later period they would carry *natural* temporal drift and confuse the classifier — a real finding worth stating in the methodology. The classifier is trained on injected drift at 5% and 10% rates and evaluated at 20% and 30% with different seeds, which reduces (not removes) the circularity of training and testing on synthetic drift.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../src'))
import numpy as np, pandas as pd
from preprocessing import temporal_split
from injection import inject_schema_drift
import module2_drift as m2
DATA_PATH = '../data/HI-Small_Trans.csv'

In [ ]:
cols=['Timestamp','Amount Paid','Amount Received','Receiving Currency','Payment Currency','Payment Format']
df = pd.read_csv(DATA_PATH, usecols=cols)
ref, inc = temporal_split(df, reference_days=3, max_days=10)
ref_pool = ref.reset_index(drop=True)
reference = ref_pool.sample(50_000, random_state=42).reset_index(drop=True)
print('reference:', reference.shape[0], '| reference pool (days 1-3):', ref_pool.shape[0])

## Step 1 — Structural check
Deterministic: catches dropped/added columns and dtype changes.

In [ ]:
batch = ref_pool.sample(20_000, random_state=1).reset_index(drop=True)
print('clean vs clean :', m2.structural_check(reference, batch))
d_drop,_ = inject_schema_drift(batch, drop_column='Payment Format')
print('dropped column :', m2.structural_check(reference, d_drop))
d_type,_ = inject_schema_drift(batch, dtype_change=('Amount Paid', str))
print('dtype change   :', m2.structural_check(reference, d_type))

## Step 2 — Statistical check
KS for numeric, Chi-squared for categorical. Here we shift the amount distribution and confirm only the amount columns flag.

In [ ]:
drifted = m2.inject_distribution_shift(batch, column='Amount Paid', rate=0.20, seed=7)
sc = m2.statistical_check(reference, drifted)
rows=[]
for col,v in sc.items():
    stat = v.get('ks_stat') if v['type']=='numeric' else v.get('chi2_stat')
    rows.append([col, v['type'], round(stat,3), format(v['p_value'],'.2g'), v['drifted']])
pd.DataFrame(rows, columns=['column','type','stat','p_value','drifted'])

## Step 3 — Random Forest drift classifier
Train at 5%/10%, evaluate at 20%/30% (different seeds). Clean batches from the reference pool.

In [ ]:
Xtr, ytr = m2.build_drift_dataset(reference, ref_pool, rates=[0.05,0.10], batch_size=20_000, n_per_rate=20, seed=42)
Xte, yte = m2.build_drift_dataset(reference, ref_pool, rates=[0.20,0.30], batch_size=20_000, n_per_rate=20, seed=999)
clf = m2.train_drift_classifier(Xtr, ytr)
res = m2.evaluate_drift_classifier(clf, Xte, yte)
print('train batches:', len(ytr), '| eval batches:', len(yte))
pd.DataFrame([{k:round(v,3) for k,v in res.items() if k!='confusion_matrix'}])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay(np.array(res['confusion_matrix']), display_labels=['clean','drifted']).plot(cmap='Blues')
plt.title('Module 2 — drift classifier confusion matrix')
plt.tight_layout(); plt.savefig('../results/module2_confusion.png', dpi=120); plt.show()

## Reading the results
- **Step 1** catches structural drift exactly (it is a direct comparison, no ML needed).
- **Step 2** flags the shifted amount columns and correctly leaves the unchanged currencies alone.
- **Step 3** separates injected drift from clean batches with high F1. Because this is a controlled synthetic experiment, a high score is expected and must be reported honestly as such, with the circularity limitation stated. The value is that the three-step design detects all four drift types.